# Raw Dataset Integrity Validation

This notebook validates the CAMELS-SIMBA raw Rockstar `hlist` files before graph construction and model training. The goal is to verify that the raw folders used in the thesis are complete, consistently named, and suitable for building halo graph datasets for `Omega_m` prediction.

This validation is intentionally read-only. It does not download, delete, or modify raw data. It only inspects file names, universe IDs, snapshot coverage, duplicate universe/snapshot pairs, suspicious tiny files, hard-link metadata, and storage usage.

## 1. Expected Dataset Structure

Each raw dataset folder should contain five selected snapshots per universe:

```text
0.20000, 0.25000, 0.51209, 0.75065, 1.00000
```

| Dataset | Expected universes | Expected files | Universe range |
|---|---:|---:|---|
| `CAMELS_SIMBA_100U` | 100 | 500 | `LH_0`-`LH_99` |
| `CAMELS_SIMBA_200U` | 200 | 1000 | `LH_0`-`LH_199` |
| `CAMELS_SIMBA_500U` | 500 | 2500 | `LH_0`-`LH_499` |
| `CAMELS_SIMBA_750U` | 750 | 3750 | `LH_0`-`LH_749` |
| `CAMELS_SIMBA_1000U` | 1000 | 5000 | `LH_0`-`LH_999` |

The setup cell below imports the required libraries, finds the repository root robustly, and creates the output directory for validation summaries.

In [1]:
from pathlib import Path
import re
import os
from collections import Counter, defaultdict

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Find repository root from either repo root or a nested notebook directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing src/ and data/.")


REPO_ROOT = find_repo_root()
RAW_ROOT = REPO_ROOT / "data" / "raw"
OUT_DIR = REPO_ROOT / "outputs" / "raw_dataset_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SNAPSHOTS = ["0.20000", "0.25000", "0.51209", "0.75065", "1.00000"]
FILENAME_RE = re.compile(r"^LH_(\d+)_hlist_([0-9]+\.[0-9]{5})\.list$")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

print(f"Repository root: {REPO_ROOT}")
print(f"Raw data root:   {RAW_ROOT}")
print(f"Output folder:   {OUT_DIR}")

Repository root: <REPO_ROOT>
Raw data root:   <REPO_ROOT>/data/raw
Output folder:   <REPO_ROOT>/outputs/raw_dataset_validation


## 2. Validation Checks

For each raw folder, the notebook checks:

- total Rockstar `hlist` file count
- valid filename pattern
- unique universe IDs
- first and last universe IDs
- missing universes
- extra universes
- exactly five snapshots per universe
- duplicate universe/snapshot pairs
- suspicious tiny files
- hard-link storage information

The code cell below defines a reusable `validate_raw_dataset(...)` function that performs these checks for one dataset folder and returns both summary metrics and example problem records.

In [2]:
def bytes_to_gb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)


def validate_raw_dataset(dataset_name: str, expected_n_universes: int) -> dict:
    folder = RAW_ROOT / dataset_name
    expected_universes = set(range(expected_n_universes))
    expected_snapshot_set = set(EXPECTED_SNAPSHOTS)

    result = {
        "dataset": dataset_name,
        "folder": str(folder),
        "folder_exists": folder.exists(),
        "expected_universes": expected_n_universes,
        "expected_files": expected_n_universes * len(EXPECTED_SNAPSHOTS),
        "total_files": 0,
        "valid_filename_files": 0,
        "unique_universe_count": 0,
        "first_universe_id": "missing",
        "last_universe_id": "missing",
        "first_10_universes": "",
        "last_10_universes": "",
        "missing_universe_count": expected_n_universes,
        "extra_universe_count": 0,
        "universes_with_wrong_snapshot_count": 0,
        "duplicate_pair_count": 0,
        "bad_filename_count": 0,
        "tiny_file_count": 0,
        "total_size_gb": 0.0,
        "average_link_count": None,
        "max_link_count": None,
        "files_with_link_count_gt_1": 0,
        "pass": False,
        "warnings": [],
        "missing_universes_examples": [],
        "extra_universes_examples": [],
        "wrong_snapshot_examples": [],
        "duplicate_pair_examples": [],
        "bad_filename_examples": [],
        "tiny_file_examples": [],
    }

    if not folder.exists():
        result["warnings"].append("Folder does not exist.")
        return result

    files = sorted(folder.glob("LH_*_hlist_*.list"))
    result["total_files"] = len(files)

    parsed_pairs = []
    universe_to_snapshots = defaultdict(list)
    bad_filenames = []
    tiny_files = []
    total_size = 0
    link_counts = []

    for path in files:
        try:
            stat = path.stat()
        except FileNotFoundError:
            continue

        total_size += stat.st_size
        link_counts.append(stat.st_nlink)
        if stat.st_size < 1024:
            tiny_files.append((path.name, stat.st_size))

        match = FILENAME_RE.match(path.name)
        if not match:
            bad_filenames.append(path.name)
            continue

        universe_id = int(match.group(1))
        snapshot = match.group(2)
        parsed_pairs.append((universe_id, snapshot, path.name))
        universe_to_snapshots[universe_id].append(snapshot)

    pair_counts = Counter((uid, snap) for uid, snap, _ in parsed_pairs)
    duplicate_pairs = [(uid, snap, count) for (uid, snap), count in pair_counts.items() if count > 1]

    found_universes = set(universe_to_snapshots.keys())
    missing_universes = sorted(expected_universes - found_universes)
    extra_universes = sorted(found_universes - expected_universes)
    sorted_universes = sorted(found_universes)

    wrong_snapshot_examples = []
    wrong_snapshot_count = 0
    for universe_id in sorted(found_universes & expected_universes):
        snapshots = universe_to_snapshots[universe_id]
        snapshot_counts = Counter(snapshots)
        snapshot_set = set(snapshots)
        missing_snapshots = sorted(expected_snapshot_set - snapshot_set)
        extra_snapshots = sorted(snapshot_set - expected_snapshot_set)
        duplicate_snapshots = sorted([snap for snap, count in snapshot_counts.items() if count > 1])

        if missing_snapshots or extra_snapshots or duplicate_snapshots:
            wrong_snapshot_count += 1
            wrong_snapshot_examples.append(
                {
                    "universe_id": f"LH_{universe_id}",
                    "found_snapshots": sorted(snapshots),
                    "missing_snapshots": missing_snapshots,
                    "extra_snapshots": extra_snapshots,
                    "duplicate_snapshots": duplicate_snapshots,
                }
            )

    result.update(
        {
            "valid_filename_files": len(parsed_pairs),
            "unique_universe_count": len(found_universes),
            "first_universe_id": f"LH_{sorted_universes[0]}" if sorted_universes else "missing",
            "last_universe_id": f"LH_{sorted_universes[-1]}" if sorted_universes else "missing",
            "first_10_universes": ", ".join(f"LH_{u}" for u in sorted_universes[:10]),
            "last_10_universes": ", ".join(f"LH_{u}" for u in sorted_universes[-10:]),
            "missing_universe_count": len(missing_universes),
            "extra_universe_count": len(extra_universes),
            "universes_with_wrong_snapshot_count": wrong_snapshot_count,
            "duplicate_pair_count": len(duplicate_pairs),
            "bad_filename_count": len(bad_filenames),
            "tiny_file_count": len(tiny_files),
            "total_size_gb": bytes_to_gb(total_size),
            "average_link_count": sum(link_counts) / len(link_counts) if link_counts else None,
            "max_link_count": max(link_counts) if link_counts else None,
            "files_with_link_count_gt_1": sum(1 for count in link_counts if count > 1),
            "missing_universes_examples": [f"LH_{u}" for u in missing_universes[:20]],
            "extra_universes_examples": [f"LH_{u}" for u in extra_universes[:20]],
            "wrong_snapshot_examples": wrong_snapshot_examples[:10],
            "duplicate_pair_examples": duplicate_pairs[:10],
            "bad_filename_examples": bad_filenames[:20],
            "tiny_file_examples": tiny_files[:20],
        }
    )

    checks = [
        result["folder_exists"],
        result["total_files"] == result["expected_files"],
        result["unique_universe_count"] == expected_n_universes,
        result["missing_universe_count"] == 0,
        result["extra_universe_count"] == 0,
        result["universes_with_wrong_snapshot_count"] == 0,
        result["duplicate_pair_count"] == 0,
        result["bad_filename_count"] == 0,
        result["tiny_file_count"] == 0,
    ]
    result["pass"] = bool(all(checks))

    if result["total_files"] != result["expected_files"]:
        result["warnings"].append("Unexpected total file count.")
    if result["missing_universe_count"]:
        result["warnings"].append("Missing expected universes.")
    if result["extra_universe_count"]:
        result["warnings"].append("Found extra universes outside expected range.")
    if result["universes_with_wrong_snapshot_count"]:
        result["warnings"].append("Some universes do not have exactly the expected snapshots.")
    if result["duplicate_pair_count"]:
        result["warnings"].append("Duplicate universe/snapshot pairs found.")
    if result["bad_filename_count"]:
        result["warnings"].append("Bad filenames found.")
    if result["tiny_file_count"]:
        result["warnings"].append("Suspicious tiny files found.")

    return result

## 3. Run Integrity Validation

This section validates all five raw CAMELS-SIMBA folders and saves a compact CSV summary to `outputs/raw_dataset_validation/raw_dataset_integrity_summary.csv`.

In [3]:
DATASETS_TO_VALIDATE = {
    "CAMELS_SIMBA_100U": 100,
    "CAMELS_SIMBA_200U": 200,
    "CAMELS_SIMBA_500U": 500,
    "CAMELS_SIMBA_750U": 750,
    "CAMELS_SIMBA_1000U": 1000,
}

validation_results = [
    validate_raw_dataset(dataset_name, expected_n_universes)
    for dataset_name, expected_n_universes in DATASETS_TO_VALIDATE.items()
]

summary_columns = [
    "dataset",
    "folder_exists",
    "expected_universes",
    "unique_universe_count",
    "first_universe_id",
    "last_universe_id",
    "expected_files",
    "total_files",
    "valid_filename_files",
    "missing_universe_count",
    "extra_universe_count",
    "universes_with_wrong_snapshot_count",
    "duplicate_pair_count",
    "bad_filename_count",
    "tiny_file_count",
    "total_size_gb",
    "average_link_count",
    "max_link_count",
    "files_with_link_count_gt_1",
    "pass",
]

raw_summary_df = pd.DataFrame(validation_results)[summary_columns]
raw_summary_df.to_csv(OUT_DIR / "raw_dataset_integrity_summary.csv", index=False)

display(raw_summary_df)
print(f"Saved summary to: {OUT_DIR / 'raw_dataset_integrity_summary.csv'}")

,dataset,folder_exists,expected_universes,unique_universe_count,first_universe_id,last_universe_id,expected_files,total_files,valid_filename_files,missing_universe_count,extra_universe_count,universes_with_wrong_snapshot_count,duplicate_pair_count,bad_filename_count,tiny_file_count,total_size_gb,average_link_count,max_link_count,files_with_link_count_gt_1,pass
0,CAMELS_SIMBA_100U,True,100,100,LH_0,LH_99,500,500,500,0,0,0,0,0,0,2.015083,5.000000,5,500,True
1,CAMELS_SIMBA_200U,True,200,200,LH_0,LH_199,1000,1000,1000,0,0,0,0,0,0,4.083980,4.500000,5,1000,True
2,CAMELS_SIMBA_500U,True,500,500,LH_0,LH_499,2500,2500,2500,0,0,0,0,0,0,10.308762,3.600000,5,2500,True
3,CAMELS_SIMBA_750U,True,750,750,LH_0,LH_749,3750,3750,3750,0,0,0,0,0,0,15.540156,3.066667,5,3750,True
4,CAMELS_SIMBA_1000U,True,1000,1000,LH_0,LH_999,5000,5000,5000,0,0,0,0,0,0,20.627465,2.550000,5,3750,True


Saved summary to: <REPO_ROOT>/outputs/raw_dataset_validation/raw_dataset_integrity_summary.csv


## 4. Problem Report

This section prints a concise PASS/FAIL report for each folder. If a dataset has missing universes, malformed filenames, duplicated universe/snapshot pairs, or suspicious tiny files, the first examples are shown.

In [4]:
for result in validation_results:
    print("=" * 90)
    print(result["dataset"])
    print("=" * 90)

    if result["pass"]:
        print(
            "PASS: expected universes and snapshots are present; "
            "no duplicate pairs, bad filenames, or suspicious tiny files were found."
        )
        continue

    print("FAIL or WARNING")

    for warning in result["warnings"]:
        print(f"- {warning}")

    problem_fields = [
        ("Missing universe examples", result["missing_universes_examples"]),
        ("Extra universe examples", result["extra_universes_examples"]),
        ("Wrong snapshot examples", result["wrong_snapshot_examples"]),
        ("Duplicate universe/snapshot pair examples", result["duplicate_pair_examples"]),
        ("Bad filename examples", result["bad_filename_examples"]),
        ("Tiny file examples", result["tiny_file_examples"]),
    ]

    for title, examples in problem_fields:
        if examples:
            print(f"\n{title}:")
            print(examples[:5])

CAMELS_SIMBA_100U
PASS: expected universes and snapshots are present; no duplicate pairs, bad filenames, or suspicious tiny files were found.
CAMELS_SIMBA_200U
PASS: expected universes and snapshots are present; no duplicate pairs, bad filenames, or suspicious tiny files were found.
CAMELS_SIMBA_500U
PASS: expected universes and snapshots are present; no duplicate pairs, bad filenames, or suspicious tiny files were found.
CAMELS_SIMBA_750U
PASS: expected universes and snapshots are present; no duplicate pairs, bad filenames, or suspicious tiny files were found.
CAMELS_SIMBA_1000U
PASS: expected universes and snapshots are present; no duplicate pairs, bad filenames, or suspicious tiny files were found.


## 5. Hard-Link Verification

Hard links were used so larger raw folders can reuse existing raw files without duplicating disk usage. If two paths have the same device and inode number, they refer to the same physical file content. A link count greater than one means the file has multiple directory entries.

In [5]:
hard_link_examples = [
    RAW_ROOT / "CAMELS_SIMBA_500U" / "LH_0_hlist_1.00000.list",
    RAW_ROOT / "CAMELS_SIMBA_750U" / "LH_0_hlist_1.00000.list",
    RAW_ROOT / "CAMELS_SIMBA_1000U" / "LH_0_hlist_1.00000.list",
]

hard_link_rows = []
for path in hard_link_examples:
    if path.exists():
        stat = path.stat()
        hard_link_rows.append(
            {
                "path": str(path.relative_to(REPO_ROOT)),
                "exists": True,
                "device": stat.st_dev,
                "inode": stat.st_ino,
                "link_count": stat.st_nlink,
                "size_bytes": stat.st_size,
            }
        )
    else:
        hard_link_rows.append(
            {
                "path": str(path.relative_to(REPO_ROOT)),
                "exists": False,
                "device": None,
                "inode": None,
                "link_count": None,
                "size_bytes": None,
            }
        )

hard_link_df = pd.DataFrame(hard_link_rows)
display(hard_link_df)

if hard_link_df["exists"].all() and hard_link_df["device"].nunique() == 1 and hard_link_df["inode"].nunique() == 1:
    print("These example paths share the same device and inode, so they are hard links to the same file content.")
else:
    print("These example paths do not all share the same inode/device, or at least one file is missing.")

,path,exists,device,inode,link_count,size_bytes
0,data/raw/CAMELS_SIMBA_500U/LH_0_hlist_1.00000....,True,64526,2130288,5,4437772
1,data/raw/CAMELS_SIMBA_750U/LH_0_hlist_1.00000....,True,64526,2130288,5,4437772
2,data/raw/CAMELS_SIMBA_1000U/LH_0_hlist_1.00000...,True,64526,2130288,5,4437772


These example paths share the same device and inode, so they are hard links to the same file content.


## 6. Storage Summary

This section estimates storage use for raw data, processed graph datasets, experiment outputs, and the full repository. The summary is saved to `outputs/raw_dataset_validation/storage_summary.csv`.

In [6]:
def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for item in path.rglob("*"):
        if item.is_file():
            try:
                total += item.stat().st_size
            except FileNotFoundError:
                pass
    return total


storage_targets = {
    "data/raw": REPO_ROOT / "data" / "raw",
    "data/processed": REPO_ROOT / "data" / "processed",
    "experiments": REPO_ROOT / "experiments",
    "full_repository": REPO_ROOT,
}

storage_rows = []
for name, path in storage_targets.items():
    size_bytes = directory_size_bytes(path)
    storage_rows.append(
        {
            "name": name,
            "path": str(path.relative_to(REPO_ROOT)) if path != REPO_ROOT else ".",
            "size_bytes": size_bytes,
            "size_gb": bytes_to_gb(size_bytes),
        }
    )

storage_df = pd.DataFrame(storage_rows)
storage_df.to_csv(OUT_DIR / "storage_summary.csv", index=False)

display(storage_df)
print(f"Saved storage summary to: {OUT_DIR / 'storage_summary.csv'}")

,name,path,size_bytes,size_gb
0,data/raw,data/raw,56452960486,52.575917
1,data/processed,data/processed,23941625263,22.297376
2,experiments,experiments,13161529036,12.257629
3,full_repository,.,94342349850,87.863160


Saved storage summary to: <REPO_ROOT>/outputs/raw_dataset_validation/storage_summary.csv


## 7. Thesis Interpretation

This validation notebook verifies the raw CAMELS-SIMBA folders used before graph construction and model training.

When the validation table reports PASS for all folders, the raw data support the following thesis interpretation:

- The raw CAMELS-SIMBA folders were validated for completeness and naming consistency.
- The largest raw dataset provides 1000 unique universes.
- Each universe has the expected five selected snapshots.
- No duplicate universe/snapshot pairs were detected.
- No suspicious tiny raw files were detected.
- This supports that later graph model results are not caused by missing, duplicated, or malformed raw halo catalogue files.